# Finite state automata

## Deterministic finite-state automaton

A deterministic finite-state automaton (DFA) is a *finite-state machine* (a mathematical model of computation) that accepts or rejects a given string of symbols by progressing through a state sequence uniquely defined by the string. 

Formally, a DFA is defined as a 5-tuple  ($Q, \Sigma, \delta, q_0, F$) where:
 - $Q$ is a finite set of states
 - $\Sigma$ is a finite alphabet
 - $\delta : Q \times \Sigma \to Q$ is the transition function
 - $q_0 \in Q$ is the start state
 - $F \subseteq Q$ is the set of accept states

![DFA example](dfa_example.png)

*This DFA example accepts the strings ending in 1, with:*

 - $Q = \{q_1, q_2\}$,
 - $\Sigma = \{0,1\}$, 
 - $\delta(q_1,0) = q_1,\quad \delta(q_1,1) = q_2,\quad \delta(q_2,0) = q_1,\quad \delta(q_2,1) = q_2$,
 - $q_0 = q_1$,
 - $F = \{q_2\}$



## Regular languages

A language is simply a set of strings over an alphabet.

The DFA $M$ recognises the language $L$ if $L = \{\, w \mid M \text{ accepts } \,w\}$. 
 - The language $L$ is the set of all strings that the DFA $M$ accepts

A regular language is any language that can recognised by some DFA.
 - If you can build a DFA that accepts exactly the strings in that language, then that language is regular

### Regular expressions

Regular expressions are another way of describing regular languages, using algebra-like rules instead of state diagrams. Regular expressions and DFAs have exactly the same expressive power.

Regular expressions are built using some basic operations, called regular operations, on languages:
 - Boolean (set-theoretic) operations - general set operations can be applied to languages, since languages are sets of strings
   - Union: $A \cup B = \{\, x \mid x \in A \text{ or } x \in B \,\}$
   - Intersection: $A \cap B = \{\, x \mid x \in A \text{ and } x \in B \,\}$
   - Difference: $A \setminus B = \{\, x \mid x \in A \text{ and } x \notin B \,\}$
   - Complement: $\overline{A} = \Sigma^{*} \setminus A$
 - Language theory specific operations - special to formal languages and regular expressions
   - Concatenation: $A \circ B = \{\, xy \mid x \in A \text{ and } y \in B \,\}$
   - Star: $A^{*} = \{\, x_1 x_2 \dots x_k \mid k \ge 0 \text{ and } x_i \in A \text{ for every } i,\ 1 \le i \le k \,\} = \bigcup_{i=0}^{\infty} A^{i}$
     - in other words, $A^{*}$ is the set of all finite strings obtained by concatenating zero or more strings from $A$, allowing repetition

The formal definition of a regular expression (RE) is inductive (recursive) - there are initial REs, and new REs can be obtained from old ones by means of regular operations.

$R$ is a regular expression over the alphabet $\Sigma$ if $R$ is any of:
1. $a$, for some $a \in \Sigma$,
2. $\varepsilon$ (the empty string),
3. $\emptyset$ (the empty language),
4. $(R_1 \cup R_2)$, where $R_1$ and $R_2$ are regular expressions,
5. $(R_1 \circ R_2)$, where $R_1$ and $R_2$ are regular expressions, and
5. $(R_1^{*})$, where $R_1$ is a regular expression.

Note:
- the concatenation symbol can be omitted, i.e. $R_1 R_2$ means $R_1 \circ R_2$
- parenthesis can be omitted, bearing in mind the precedence order from highest to lowest is $^*$, $\circ$, $\cup$, i.e. $01 \cup 1^*$ means $((01) \cup (1^{*}))$

## Combining automata

Automata can be combined in order to implement an operation on languages. This is to build a new DFA that recognises a new language, specifically one obtained by applying an operation (e.g. union) to languages that are already recognised by DFAs.

For example, given $L_1$ is recognised by $M_1 = (Q_1, \Sigma, \delta_1, q_1, F_1)$ and $L_2$ is recognised by $M_2 = (Q_2, \Sigma, \delta_2, q_2, F_2)$, we can combine $M_1$ and $M_2$ into a new automaton M that would recognise $L_1 \cup L_2$.

It will not work to first simulate the input on $M_1$, then simulate it on $M_2$, and accept if either $M_1$ or $M_2$ or both accept, since after $M_1$ has run on the input, the input is exhausted - A DFA cannot 'rewind' the input in order to run $M_2$.

The solution is to run $M_1$ and $M_2$ in parallel, using pairs of states (a state for each DFA). The transition function can be defined as below:
![Transition function](parallel_transition_function.png)

Formally, the components of the new automaton $M = (Q, \Sigma, \delta, q_0, F)$ are:
- $Q = Q_1 \times Q_2$
- $\delta((s,u), a) = (\delta_1(s,a), \delta_2(u,a)), \quad \forall s \in Q_1, u \in Q_2, a \in \Sigma$
- $q_0 = (q_1, q_2)$
- $F = \{ (s,u) \in Q \mid s \in F_1 \text{ or } u \in F_2 \}$

## Concatenation

The above method for combining automata doesn't work for recognising concatenation. 

$w \in L_1 \circ L_2$ only if there are words $x$ and $y$ such that $w = xy$, $x \in L_1$ and $y \in L_2$.

If $L_1$ is recognised by $M_1 = (Q_1, \Sigma, \delta_1, q_1, F_1)$ and $L_2$ is recognised by $M_2 = (Q_2, \Sigma, \delta_2, q_2, F_2)$, then to recognise $w$, $M_1$ would need to run on some prefix of $w$, and $M_2$ on the rest. The problem lies in finding the break point for this - the only way to do this is to guess it non-deterministically. 

In order to implement concatenation, non-deterministic finite automatons are required.

# Non-deterministic finite automaton (NFA)

A nondeterministic finite-state automaton (NFA) is a finite-state machine that accepts or rejects a given string of symbols by progressing through one or more possible state sequences defined by the string, where at least one such sequence leads to an accepting state.

Formally, an NFA is defined as a 5-tuple  ($Q, \Sigma, \delta, q_0, F$) where:
 - $Q$ is a finite set of states
 - $\Sigma$ is a finite alphabet
 - $\delta : Q \times (\Sigma \cup \{\varepsilon\}) \to \mathcal{P}(Q)$ is the transition function
 - $q_0 \in Q$ is the start state
 - $F \subseteq Q$ is the set of accept states

The only difference between the DFA and NFA definitions is in the transition function:
- in a DFA, the transition function maps a state and an input symbol to exactly one next state
- in an NFA, the transition function the transition function maps a state and an input symbol (or $\varepsilon$) to a set of possible next states, allowing multiple or zero transitions. Here, $\varepsilon$ represents the empty string, meaning the automaton can move to the next state without consuming any input symbol

![NFA example](nfa_example.png)

From this example, three potential problems can be identified:
- there is more than one next possible state (in $q_1$, an input $1$ can lead to $q_1$ or $q_2$)
- $\varepsilon$-transition (in $q_2$, if the next symbol is $0$, and $\varepsilon$ is used to transition to $q_3$, then the $0$ will not be consumed, which may lead to rejection, even though the string may contain the substring $101$)
- no 'next' state (in $q_3$, an input 0 has no next state, so the computation hangs, meaning it ends before the input has been exhausted)
  - in this case, this is treated as rejection
 
These are not actually problems, due to the nature of how NFAs work:
- an NFA can have multiple possible next states for the same input symbol and $\varepsilon$-transitions that move without consuming input
- conceptually, the NFA explores all possible paths simultaneously
- the input is accepted if any path reaches an accepting state after consuming the entire input
- if all paths fail, the input is rejected

Since every possibility is tried: 
- multiple next states are fine - each path is explored
- $\varepsilon$-transitions are fine - there's also a path where the $\varepsilon$-transition isn't taken, so the input isn't lost
- paths that 'hang' (no next state) are fine - other valid paths will continue and may still lead to acceptance

Using the NFA example above:

![All possible computation](nfa_all_computations.png)

### Computation of an NFA

Let $M = (Q, \Sigma, \delta, q_0, F)$ be an NFA and $w = w_1 w_2 \dots w_n$ be a word over $\Sigma \cup \{\varepsilon\}$.

$M$ accepts $w$ if there exists a sequence of states $r_0, r_1, r_2, \dots, r_n$ satisfying the following conditions:
- $r_0 = q_0$
- $r_{i+1} \in \delta(r_i, w_{i+1})$ for every $i$, $0 \le i \le n-1$
- $r_n \in F$

### DFA vs NFA computation
- In DFA, there is a unique computation (path). It either accepts or rejects.
- In NFA, there are, in general, many computational paths. The NFA accepts if at least one computation accepts. The NFA rejects only if all the computations reject.

## Converting an NFA into a DFA

In an NFA, at any point, the machine could be in multiple possible states at the same time because of non-determinism. A DFA cannot be in multiple states at once - it must have exactly one current state. Therefore, in order to convert an NFA into a DFA, make each state of the DFA represent a subset of states of the NFA.

In the converted DFA:
- each DFA state is a set of NFA states
- the DFA start state is the set containing the NFA start state and all states reachable from it via $\varepsilon$-transitions
- the DFA transition function is defined as:
  - For a DFA state $R \subseteq Q_{NFA}$ and input symbol $a \in \Sigma$:
    1. Look at all NFA states in $R$
    2. For each state $r \in R$, see where the NFA can go on input $a$
    3. Take the union of all these resulting states
    4. Add all states reachable via $\varepsilon$-transitions
       - This gives the next DFA state
- a DFA state is accepting if it contains any NFA accepting state

An example of how a the transition function of the DFA can be defined is given below:

![NFA TO DFA transition function definition](nfa_dfa_trans_func.png)

More formally, let $N = (Q, \Sigma, \delta, q_0, F)$ be an NFA recognising some language $L$. We construct a DFA $M = (Q', \Sigma, \delta', q_0', F')$ recognising the same language $L$ as follows:

Define the $\varepsilon$-closure of a set of states:
$$ E(R) = \{ q \in Q \mid q \text{ is reachable from some } r \in R \text{ through a number of $\varepsilon$-transitions} \}. $$
(Note that $R \subseteq E(R)$ because each state is reachable from itself using zero $\varepsilon$-transitions.)

1. $Q' = \mathcal{P}(Q)$, the powerset (all possible subsets) of $Q$.
2. For $R \in Q'$ and $a \in \Sigma$, define
    $$ \delta'(R, a) = \bigcup_{r \in R} E(\delta(r,a)) = \{ q \in Q \mid q \in E(\delta(r,a)) \text{ for some } r \in R \}$$
3. $q_0' = E(\{q_0\})$.
4. $F' = \{ R \in Q' \mid R \cap F \neq \emptyset \}$.

An example of the conversion of an NFA to a DFA is shown below:

![NFA to DFA conversion](nfa_to_dfa.png)

Note that the red states are redundant and can be removed, as there are no ways to reach those states from the start state. The {} state is necessary as it collects all the paths that would 'hang' in the NFA. 

Proving that every NFA has an equivalent DFA has the following corollary:
- A language is regular iff some NFA recognises it

## Closure under regular expressions

A class of languages (in this case the class is regular languages) is closed under an operation if applying that operation to languages in the class produces another language in the class.

Therefore, to prove the closure of the regular operations (union, concatenation and star), it must be shown that:
- if $L_1$ and $L_2$ are regular, then $L_1 \cup L_2$, $L_1 L_2$ and $L_1^*$ are also regular

In other words, it must be proven that if $L_1$ and $L_2$ can be recognised by some DFA, then $L_1 \cup L_2$, $L_1 L_2$ and $L_1^*$ can also be recognised by some DFA.

However, since we now know that every NFA has an equivalent DFA, this can be proven by proving that $L_1 \cup L_2$, $L_1 L_2$ and $L_1^*$ can be recognised by some NFA.

### Union
1. Assume $L_1$ and $L_2$ are regular
2. Therefore, there exist DFAs $M_1$ and $M_2$ recognising them
3. Construct an NFA by:
   - adding a new start state
   - adding $\varepsilon$ transitions from this new start state to the start states of $M_1$ and $M_2$
4. This NFA recognises $L_1 \cup L_2$

![NFA recognising union](union_closure.png)

### Concatenation
1. Assume $L_1$ and $L_2$ are regular
2. Therefore, there exist DFAs $M_1$ and $M_2$ recognising them
3. Construct an NFA by:
   - adding $\varepsilon$ transitions from each accepting state of $M_1$ to the start state of $M_2$
   - making the accepting states only the original accepting states of $M_2$ 
4. This NFA recognises $L_1 L_2$

![NFA recognising concatenation](concatenation_closure.png)

### Star
1. Assume $L_1$ is regular
2. Therefore, there exists a DFA $M_1$ recognising it
3. Construct an NFA by:
   - adding a new start state which is also an accepting state
   - adding $\varepsilon$ transitions:
     - from the new start state to the old start state
     - from each old accepting state back to the old start state
4. This NFA recognises $L_1^*$

![NFA recognising star](star_closure.png)

## Equivalence of finite automata (FA) and regular expression (RE)

Theorem: A language is regular iff some regular expression describes it.

To prove this theorem, first prove that given a regular expression $R$, an FA recognising $L(R)$ can be constructed. 

This can easily be proved using structural induction:
- all initial REs can be implemented by NFAs
- every RE can be built from the initial REs using the regular operations, and we have proven that the class of regular languages is closed under regular operations

The example below shows the construction of an NFA for the regular expression $(ab \cup a)^*$:

![construction of NFA from regular expression](nfa_from_re.png)

However, proving the reverse - given a DFA $M$, construct a RE that recognises $L(M)$ - is harder. 


## Generalised non-deterministic finite automaton (GNFA)

A generalised non-deterministic finite automaton (GNFA) is an NFA with some additional characteristics:
- a single start state with no incoming edges
- a single accept state with no outgoing edges
- every transition is labelled with a regular expression

An example of a GNFA is shown below:

![GNFA example](gnfa_example.png)

A GNFA can be to prove DFA $\rightarrow$ RE, since a DFA can first be converted to a GNFA, which can then be used to produce a regular expression describing the language.

To convert from a DFA to a GNFA:
1. Add a new start state $s$, with a $\varepsilon$ transition to the old start state
2. Add a new accept state $f$, with $\varepsilon$ transitions from all old accept states
3. Replace each DFA transition with a regex
4. Add missing edges and label them with $\emptyset$

![DFA to GNFA conversion](dfa_to_gnfa.png)

Now, to get from GNFA to a regular expression, the internal states must be eliminated one-by-one, until there is a just a single transition from the start state to the accept state, at which point the regular expression for the transition fully describes the language. 

To eliminate a state $r$:

- For every pair of states $u, v \neq r$, update:
$$ R_{uv} = R_{uv} \cup R_{ur}(R_{rr})^*R_{rv} $$
- Then remove $r$ and all its edges
- Note that the case where $u \equiv v$ must also be considered

![Internal state elimination](state_elimination.png)

State elimination can be used to find a single regular expression describing the set of all strings that take a finite automaton from one state to another.

# Non-regular languages
A non-regular language is a language which is not recognised by an DFA.

To prove that a language $N$ is not regular, use proof by contradiction:
1. Assume that $N$ is regular
2. There must be some DFA that recognises $N$
3. Use an adversary argument to show that any such DFA must fail
4. This leads to a contradiction, so the assumption was false
5. Therefore, $N$ is not regular

The pumping lemma is a common formal adversary argument used to carry out Step 3 by exploiting the finite number of states in any DFA.

## Pumping lemma

The pumping lemma states that:
- For every regular language $L$, there is a number $p$ (the "pumping length" of $L$) such that every word $w \in L$ of length at least $p$ can be divided into three parts, $w = xyz$, satisfying the following conditions:
  1. $x y^i z \in L$ for every $i \in \mathbb{N}$
  2. $y$ is a non-empty string
  3. the length of $xy$ is not greater than $p$

Informally, this means that if $L$ is regular, then every long string can be 'pumped' and the result stays in $L$ - 'pumped' meaning that the string can be split into three pieces, and the middle piece repeated an arbitrary number of times. 

### Proof
The pumping lemma can be proved as follows:
1. Let the DFA $M$ recognise the language $A$. Let $p$ be the number of states in $M$. Pick $w \in A$ where $|w| \geq p$
2. Since $|w| \geq p$, when $w$ is input into $M$, $M$ is guaranteed to repeat a state
3. Let the first occurrence of the repeated state be $q_r$ and the second occurrence be $q_s$, so $0 \leq r \lt s \leq p$
4. Let:
   - $x = a_1 a_2 \dots a_r, \quad$
   - $y = a_{r+1} \dots a_s, \quad$
   - $z = a_{s+1} \dots a_{|w|}$
    
   where $s = a_1 a_2 \dots a_{|w|}$

   Then: 
   - $|xy| = s \leq p$
   - $|y| = s - r \gt 0$
5. Because $y$ corresponds to a loop:
   - $\delta(q_r,y) = q_r$
   - repeating $y$ any number of times keeps the DFA in the same state $q_s$
   - so reading $x y^i z$ always leads to the same final state as $w$
   - therefore, $x y^i z \in L \quad \forall i \in \mathbb{N}$

Proving that a language does not satisfy the pumping lemma proves that the language is not regular, since all regular languages must satisfy it.

This is typically done by choosing a 'bad' input string, and pumping it up, then proving that the pumped up string is not in the language. 

For example, proving that the language $L=\{a^n b^n \, | \, n \geq 0\}$ is irregular:
- this language requires any number of $a$'s followed by the same number of $b$'s
- if $L$ were regular, then there exists a pumping length $p$
- any string $s \in L$ longer than $p$ can be written as $s = xyz$, with:
  - $|xy| \leq p$
  - $|y| \gt 0$
  - $x y^i z \in L$ for all $i \geq 0$
- pick the string $s = a^p b^p$
- the first $p$ symbols of $s$ are all $a$'s, and the pumping lemma forces $|xy| \leq p$
- therefore $y = a^k$ for some $k \gt 0$
- choose for example $i = 0$ for the third condition of the pumping lemma
- $x y^i z = x y^0 z = x z = a^{p-k}b^p$
- this string has fewer $a$'s than $b$'s, so $x y^0 z \notin L$
- this contradicts the pumping lemma's third condition: $x y^i z \in L$ for all $i \geq 0$
- therefore, the assumption that $l$ is regular must be false.

Note that while failing the pumping lemma proves non-regularity, satisfying it does not prove regularity.

# Büchi automata

Büchi automata are a type of finite automaton for infinite strings.

They are essentially non-deterministic finite automata equipped with a special acceptance condition for infinite inputs.

An infinite input over the alphabet $\Sigma$ is denoted using $\Sigma^\omega$. A finite input over the same alphabet is denoted using $\Sigma^*$.

A DFA or NFA accepts a finite word $w_1 w_2 \ldots w_n \in \Sigma^*$ if there is a sequence of states $r_0, r_1, \ldots, r_n$ satisfying the following conditions:
- $r_0 = q_0$
- $(r_i, w_{i+1}, r_{i+1}) \in \Delta$ for every $i$, $0 \leq i \leq n-1$
  - this means that for each position in the input word, when the automaton is in state $r_i$, and the next input symbol is $w_{i+1}$, the automaton is allowed (by its transition relation $\Delta$) to move to state $r_{i+1}$.
- $r_n \in F$

Whereas, a Büchi automaton accepts $w_1 w_2 \ldots \in \Sigma^\omega$ if there is a sequence of states $r_0, r_1, \ldots \in Q^\omega$ satisfying the following conditions:
- $r_0 = q_0$
- $(r_i, w_{i+1}, r_{i+1}) \in \Delta$ for every $i$, $i \geq 0$
- there are infinitely many $r_i$s in $F$

In other words, a Büchi automaton accepts an infinite string $w$ if and only if there exists a run of the automaton on $w$ that visits at least one accepting state in $F$ infinitely often.

## $\omega$-regular languages

An $\omega$-regular language/expression is built upon regular languages, using the following operations:
- $A \cup B$, where both $A$ and $B$ are $\omega$-regular
- $AB$, where $A$ is regular and $B$ is $\omega$-regular
- $A^\omega$, which is $\{a_1 a_2 \ldots \, | \, a_i \in A\}$, i.e. an infinite sequence of words from $A$, where $A$ is regular and doesn't contain the empty word

A language $L \subseteq \Sigma^\omega$ is $\omega$-regular if and only if it is recognized by some non-deterministic Büchi automaton.

To prove that every language accepted by a Büchi automaton can be written using $\omega$-regular expressions:
1. Fix a Büchi automaton: $\mathcal{B} = (Q, \Sigma, \Delta, q_0, F)$
   - a word $\omega \in \Sigma^\omega$ is accepted by $\mathcal{B}$ iff there exists one run $r_0, r_1, r_2, \ldots$ such that:
     - $r_0 = q_0$
     - transitions follow $\omega$, , i.e. $(r_i, w_{i+1}, r_{i+1}) \in \Delta$ for all $i \geq 0$
     - some accepting state $q \in F$ occurs infinitely often in the run
2. There exists at least one accepting state $q \in F$ such that the run visits $q$ infinitely many times
3. Fix that particular state $q$
   - the run looks like $q_0 \rightarrow \ldots \rightarrow q \rightarrow \ldots \rightarrow q \rightarrow \ldots \rightarrow q \rightarrow \ldots$
4. The run can be split into two parts
   - the finite prefix of the input word that takes the automaton: $q_0 \xrightarrow{u} q$
     - the set of all such finite words is exactly $L_{q_0 q}$, which is a regular language
   - An infinite sequence of finite loops from $q$ back to $q$
     - each return corresponds to a finite word that takes: $q \xrightarrow{v} q$
     - all such words form the regular language $L_{qq}$
     - since this happens infinitely often, the rest of the input word is $v_1, v_2, v_3, \ldots$, with $v_i \in L_{qq}$
     - this is exactly $L_{qq}^\omega$
5. Therefore, every accepted word has the form $u v_1 v_2 v_3 \cdots$ with $u \in L_{q_0 q}$ and $v_i \in L_{qq}$
   - the set of words accepted via this particular accepting state $q$ is $L_{q_0 q} L_{qq}^\omega$
6. The repeating accepting state can be any state $q \in F$, so the full language accepted by the automaton is:
   $$\bigcup_{q \in F} L_{q_0 q} L_{qq}^\omega$$
7. This expression uses only regular languages, union, concatenation, and the $\omega$-operation. Therefore, every language accepted by a Büchi automaton is $\omega$-regular.